# F1 · Which scaler for which model, with and without outliers

**Type**: Compare · **Level**: ●●● challenge · **Time**: about 50 minutes · **extended path**

## Objective

Compare five preprocessing choices (none, Standard, MinMax, Robust, Quantile) on four models (KNN, logistic
regression, tree, SVM) first on clean data and then with a small number of severe outliers, and answer with numbers "which scaler
is needed by whom".

## Dataset

`ds.loans(with_missing=False)`: the four numeric columns (`age`, `income`, `job_tenure`, `amount`) and the target `defaulted`.
`X` is the clean version. In `X_out` 2% of the rows (30 out of 1500, chosen at random) have `income` and `amount` multiplied by 100: a
unit-of-measure error (cents instead of euros), typical of real data.

## Educational purpose

Stop applying `StandardScaler` out of habit. Every model reacts differently to scaling, and every scaler reacts
differently to outliers: the right choice is a **pair** (model, scaler), and you evaluate it in cross-validation.

## Task

1. Write `evaluate_all(X, y)`: for every model and every scaler build `make_pipeline(scaler, model)` and compute the mean AUC in cross-validation (`cv` provided, `scoring="roc_auc"`). Return a DataFrame with the scalers on the rows and the models on the columns. Use the dictionaries `MODELS` and `SCALERS` already defined.
2. Save `clean_table = evaluate_all(X, y)` and `outlier_table = evaluate_all(X_out, y)`.
3. Save in `tree_invariant` `True` if, in **both** tables, the AUC of the tree varies by less than 0.01 across all the scalers.
4. Compute `outlier_loss`: a DataFrame (`clean_table − outlier_table`) and find `most_fragile_scaler` (the scaler with the largest mean loss over the three scale-sensitive models: `knn`, `logistic`, `svm`) and `most_robust_scaler` (the one with the smallest loss).
5. Assign to `best_knn_out` the name of the scaler with the highest AUC for the KNN in `outlier_table`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, QuantileTransformer, RobustScaler, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

df = ds.loans(with_missing=False)
NUM = ["age", "income", "job_tenure", "amount"]
X = df[NUM].to_numpy()
y = df["defaulted"].to_numpy()
X_out = X.copy()
outlier_rows = np.random.default_rng(0).choice(len(X), 30, replace=False)
X_out[np.ix_(outlier_rows, [1, 3])] *= 100                 # unit error on income and amount
cv = StratifiedKFold(5, shuffle=True, random_state=0)

MODELS = {"knn": lambda: KNeighborsClassifier(15), "logistic": lambda: LogisticRegression(max_iter=2000),
          "tree": lambda: DecisionTreeClassifier(max_depth=5, random_state=0), "svm": lambda: SVC(C=1.0)}
SCALERS = {"none": lambda: "passthrough", "standard": lambda: StandardScaler(), "minmax": lambda: MinMaxScaler(),
           "robust": lambda: RobustScaler(), "quantile": lambda: QuantileTransformer(n_quantiles=200, output_distribution="normal")}

## Your work

In [ ]:
def evaluate_all(X, y):
    tab = pd.DataFrame(index=list(SCALERS), columns=list(MODELS), dtype=float)
    for scaler_name, make_scaler in SCALERS.items():
        for model_name, make_model in MODELS.items():
            ...
    return tab

In [ ]:
clean_table = evaluate_all(X, y)
outlier_table = evaluate_all(X_out, y)
display(clean_table.round(3)); display(outlier_table.round(3))

In [ ]:
tree_invariant = ...
outlier_loss = ...
most_fragile_scaler = ...
most_robust_scaler = ...
best_knn_out = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Inside the double loop: `pipe = make_pipeline(make_scaler(), make_model())` and `tab.loc[scaler_name, model_name] = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc").mean()`. The mean loss over the sensitive models: `outlier_loss[["knn", "logistic", "svm"]].mean(axis=1)`; `idxmax()` and `idxmin()`.'):
    check(clean_table.shape == (5, 4) and outlier_table.shape == (5, 4), "two tables of 5 scalers x 4 models", "rows: scalers, columns: models")
    check(bool(tree_invariant) and np.ptp(clean_table["tree"]) < 0.01 and np.ptp(outlier_table["tree"]) < 0.01,
          "the tree does not depend on the scaler (variation below 0.01 in both tables)", "monotonic transformations do not change the splits")
    check(abs(clean_table.loc["standard", "knn"] - clean_table.loc["none", "knn"]) < 0.02,
          "on clean data scaling does not change the KNN: the informative features are also the ones on the larger scale",
          "on clean data KNN with and without StandardScaler should be close (less than 0.02)")
    check(outlier_table.loc["standard", "knn"] < outlier_table.loc["none", "knn"] - 0.05,
          "with outliers StandardScaler makes the KNN WORSE than not scaling at all",
          "with outliers StandardScaler should be worse than no scaler for the KNN (by more than 0.05)")
    check(outlier_table.loc["minmax", "knn"] < clean_table.loc["minmax", "knn"] - 0.03,
          "with outliers MinMax makes the KNN lose more than 0.03 of AUC", "outliers squash all the normal data with MinMax")
    check(most_fragile_scaler in ("minmax", "standard"), "the most fragile scaler is based on minimum/maximum or mean/deviation", "the most fragile is the one with the largest mean loss")
    check(most_robust_scaler in ("robust", "quantile"), "the most robust scaler is based on quantiles", "the most robust is robust or quantile")
    check(best_knn_out in ("robust", "quantile"), "with outliers the best scaler for the KNN is based on quantiles", "look at the knn column of outlier_table")
done()

---
The commented solution is in `solutions/F_compare/F1_scaler_per_model_with_and_without_outliers.ipynb`. Open it after you pass the check, or after you have really tried.